In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    sentinel_dirs = ('Step0_Data', 'Step8_HistoricalCompletion')
    nested_release_dirs = ('Global-solid-waste-prediction',)
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        for release_dir in nested_release_dirs:
            release_candidate = candidate / release_dir
            if all((release_candidate / name).is_dir() for name in sentinel_dirs):
                return release_candidate
    raise FileNotFoundError('Could not locate GitHub release root from the current working directory')

release_root = find_release_root(cwd)
step7_dir = release_root / 'Step7_ModelTraining' / '2_Results'
predictions_long_path = step7_dir / '1_predictions_all_wastes_long.csv'
output_path = release_root / 'Step8_HistoricalCompletion' / '2_Results' / '0_historical_panel_completed.csv'
WASTE_TYPES = ['AW', 'CDW', 'IW', 'MSW']

In [ ]:
wide_df = pd.read_csv(step7_dir / '0_original_wide.csv')
result_wide = wide_df.copy()
pred_long = pd.read_csv(predictions_long_path)

expected_waste_types = set(WASTE_TYPES)
actual_waste_types = set(pred_long['WasteFlag'].dropna().unique())
missing = sorted(expected_waste_types - actual_waste_types)
extra = sorted(actual_waste_types - expected_waste_types)
if missing or extra:
    raise RuntimeError(f'Unexpected WasteFlag set: missing={missing}, extra={extra}')

pred_long_key_duplicates = pred_long.duplicated(subset=['Country Code', 'Year', 'WasteFlag'], keep=False)
if pred_long_key_duplicates.any():
    duplicate_rows = pred_long.loc[pred_long_key_duplicates, ['Country Code', 'Year', 'WasteFlag']]
    raise RuntimeError(
        'Duplicate Country Code + Year + WasteFlag rows found in pred_long: '
        f"{duplicate_rows.drop_duplicates().to_dict('records')}"
    )

result_wide.shape

In [ ]:
for waste_type in WASTE_TYPES:
    part = pred_long[pred_long['WasteFlag'] == waste_type].copy()
    pred_col = f'{waste_type}_pred'
    final_col = f'{waste_type}_final'
    source_col = f'{waste_type}_source'
    obs_col = f'{waste_type}_t'
    mapping = part.loc[:, [
        'Country Code',
        'Year',
        'Predicted_Raw',
        'Final_Raw',
        'Source',
    ]].copy()
    mapping = mapping.rename(columns={
        'Predicted_Raw': pred_col,
        'Final_Raw': final_col,
        'Source': source_col,
    })
    mapping_key_duplicates = mapping.duplicated(subset=['Country Code', 'Year'], keep=False)
    if mapping_key_duplicates.any():
        duplicate_rows = mapping.loc[mapping_key_duplicates, ['Country Code', 'Year']]
        raise RuntimeError(
            f'Duplicate Country Code + Year rows found in mapping for {waste_type}: '
            f"{duplicate_rows.drop_duplicates().to_dict('records')}"
        )
    result_wide = result_wide.merge(mapping, on=['Country Code', 'Year'], how='left')
    result_wide[obs_col] = pd.to_numeric(result_wide[obs_col], errors='coerce')

if len(result_wide) != len(wide_df):
    raise RuntimeError(f'Row count changed after merge: expected={len(wide_df)}, actual={len(result_wide)}')

result_key_duplicates = result_wide.duplicated(subset=['Country Code', 'Year'], keep=False)
if result_key_duplicates.any():
    duplicate_rows = result_wide.loc[result_key_duplicates, ['Country Code', 'Year']]
    raise RuntimeError(
        'Duplicate Country Code + Year rows found in result_wide: '
        f"{duplicate_rows.drop_duplicates().to_dict('records')}"
    )

if 'IncomeGroup' in result_wide.columns:
    eth_income_mask = result_wide['Country Code'].astype(str).eq('ETH') & result_wide['IncomeGroup'].isna()
    result_wide.loc[eth_income_mask, 'IncomeGroup'] = 'Low income'

required_columns = []
for waste_type in WASTE_TYPES:
    required_columns.extend([
        f'{waste_type}_pred',
        f'{waste_type}_final',
        f'{waste_type}_source',
    ])
missing_columns = [column for column in required_columns if column not in result_wide.columns]
if missing_columns:
    raise RuntimeError(f'Missing required result columns: {missing_columns}')

result_wide.shape

In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
result_wide.to_csv(output_path, index=False)
output_path